# Transformer vs RNN/LSTM: From Scratch in PyTorch

We'll build three sequence-to-sequence models on a toy task: **sequence reversal**.

Input: `[3, 7, 1, 9, 5]` -> Output: `[5, 9, 1, 7, 3]`

This task is deceptively hard for sequential models (RNN/LSTM) because the first output token
depends on the *last* input token. A Transformer can attend to any position directly.

**Roadmap:**
1. Setup & data generation
2. PyTorch primer (training loop pattern)
3. RNN seq2seq
4. LSTM seq2seq
5. Transformer (built from scratch - Q/K/V, multi-head attention, the works)
6. Comparison

---
## Part 1: Setup & Toy Task

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import time
import math

# Auto-detect GPU (works on Colab/Kaggle with GPU, falls back to CPU locally)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
torch.manual_seed(42)

In [ ]:
# --- Task: Sequence Reversal ---
# Input:  [3, 7, 1, 9, 5, 12, 8, 15, 2, 6]
# Output: [6, 2, 15, 8, 12, 5, 9, 1, 7, 3]
#
# Vocab: integers 1..19 (we reserve 0 as a special <SOS> start-of-sequence token)
# Sequence length: 10

VOCAB_SIZE = 20   # tokens 0..19  (0 = <SOS>)
SEQ_LEN = 10
SOS_TOKEN = 0     # start-of-sequence for decoder
NUM_TRAIN = 5000
NUM_TEST = 500
BATCH_SIZE = 64

class ReversalDataset(Dataset):
    def __init__(self, num_samples):
        # Random integers in [1, VOCAB_SIZE-1] — avoid 0 (reserved for SOS)
        self.src = torch.randint(1, VOCAB_SIZE, (num_samples, SEQ_LEN))
        self.tgt = self.src.flip(dims=[1])  # reverse each sequence

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

train_dataset = ReversalDataset(NUM_TRAIN)
test_dataset = ReversalDataset(NUM_TEST)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Quick peek
src_sample, tgt_sample = train_dataset[0]
print(f'Input:  {src_sample.tolist()}')
print(f'Target: {tgt_sample.tolist()}')

---
## Part 2: PyTorch Primer

Before building anything complex, let's establish the **training loop pattern** we'll reuse throughout.

Every PyTorch model follows the same flow:
1. **Define the model** — a class inheriting from `nn.Module` with a `forward()` method
2. **Pick a loss function** — measures how wrong the predictions are
3. **Pick an optimizer** — adjusts weights to reduce loss (Adam is the go-to)
4. **Training loop** — feed data in, compute loss, backpropagate, update weights

In [ ]:
# Quick demo: the world's simplest "model" — just to show the pattern.
# This won't solve reversal, but it shows how PyTorch training works.

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, 16)         # token -> 16-dim vector
        self.linear = nn.Linear(16 * SEQ_LEN, SEQ_LEN * VOCAB_SIZE)  # flatten -> predict all positions

    def forward(self, src):
        x = self.embed(src)             # (batch, seq_len, 16)
        x = x.view(x.size(0), -1)      # (batch, seq_len * 16)  — flatten
        x = self.linear(x)              # (batch, seq_len * vocab_size)
        x = x.view(-1, SEQ_LEN, VOCAB_SIZE)  # (batch, seq_len, vocab_size)
        return x

# Instantiate
demo_model = TinyModel().to(device)
print(f'Parameters: {sum(p.numel() for p in demo_model.parameters()):,}')

In [ ]:
# The training loop pattern — memorize this shape, every model reuses it

loss_fn = nn.CrossEntropyLoss()   # standard for classification (predicting which token)
optimizer = optim.Adam(demo_model.parameters(), lr=0.001)

demo_losses = []
for epoch in range(20):
    epoch_loss = 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)

        # Forward pass
        output = demo_model(src)                    # (batch, seq_len, vocab_size)
        loss = loss_fn(output.view(-1, VOCAB_SIZE), tgt.view(-1))  # flatten for CrossEntropy

        # Backward pass
        optimizer.zero_grad()   # clear old gradients
        loss.backward()         # compute new gradients
        optimizer.step()        # update weights

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    demo_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}')

print('\nDone! This simple model won\'t solve reversal well, but the PATTERN is what matters.')

### Helper functions

We'll reuse these for training and evaluating all three real models.

In [ ]:
def train_model(model, train_loader, epochs=50, lr=0.001, label=None):
    """Train a seq2seq model and return (losses, elapsed_time)."""
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []

    start = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for src, tgt in train_loader:
            src, tgt = src.to(device), tgt.to(device)

            output = model(src, tgt)  # (batch, seq_len, vocab_size)
            loss = loss_fn(output.view(-1, VOCAB_SIZE), tgt.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        if (epoch + 1) % 10 == 0:
            print(f'  [{label or "Model"}] Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}')

    elapsed = time.time() - start
    print(f'  [{label or "Model"}] Training done in {elapsed:.1f}s')
    return losses, elapsed


def evaluate_model(model, test_loader):
    """Compute token-level and sequence-level accuracy."""
    model.eval()
    correct_tokens = 0
    total_tokens = 0
    correct_seqs = 0
    total_seqs = 0

    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt)
            preds = output.argmax(dim=-1)  # (batch, seq_len)

            correct_tokens += (preds == tgt).sum().item()
            total_tokens += tgt.numel()
            correct_seqs += (preds == tgt).all(dim=1).sum().item()
            total_seqs += tgt.size(0)

    token_acc = correct_tokens / total_tokens
    seq_acc = correct_seqs / total_seqs
    return token_acc, seq_acc


def show_predictions(model, dataset, n=5):
    """Show n example predictions."""
    model.eval()
    with torch.no_grad():
        for i in range(n):
            src, tgt = dataset[i]
            src_dev = src.unsqueeze(0).to(device)
            tgt_dev = tgt.unsqueeze(0).to(device)
            output = model(src_dev, tgt_dev)
            pred = output.argmax(dim=-1).squeeze(0)

            match = 'OK' if pred.tolist() == tgt.tolist() else 'WRONG'
            print(f'  Input:  {src.tolist()}')
            print(f'  Target: {tgt.tolist()}')
            print(f'  Pred:   {pred.tolist()}  [{match}]')
            print()

---
## Part 3: RNN Seq2Seq

An RNN processes sequences **one token at a time**, passing a hidden state forward like a game of telephone.

For seq2seq (input sequence -> output sequence), we use the **encoder-decoder** pattern:
- **Encoder**: Reads the full input, compresses it into a single hidden state vector
- **Decoder**: Takes that hidden state and generates the output sequence one token at a time

The bottleneck is clear: the *entire* input sequence gets squashed into one fixed-size vector.
For reversal, the decoder must remember the exact order of 10 tokens from that single vector.

```
Encoder:  [3] -> h1 -> [7] -> h2 -> [1] -> h3 -> ... -> [6] -> h_final
                                                                    |
Decoder:                                              h_final -> [6] -> [2] -> [15] -> ...
```

In [ ]:
class RNNSeq2Seq(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden_dim=64, num_layers=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Shared embedding for encoder and decoder
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Encoder: reads input sequence, produces hidden state
        self.encoder = nn.RNN(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)

        # Decoder: generates output sequence from hidden state
        self.decoder = nn.RNN(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)

        # Final projection: hidden state -> vocab scores
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src, tgt):
        # Encode: process full input sequence
        src_emb = self.embedding(src)          # (batch, seq_len, embed_dim)
        _, hidden = self.encoder(src_emb)       # hidden: (num_layers, batch, hidden_dim)

        # Decode: use target as input (teacher forcing)
        # Prepend SOS token, drop last token from target
        batch_size = tgt.size(0)
        sos = torch.zeros(batch_size, 1, dtype=torch.long, device=tgt.device)  # SOS = 0
        dec_input = torch.cat([sos, tgt[:, :-1]], dim=1)   # shift right

        dec_emb = self.embedding(dec_input)     # (batch, seq_len, embed_dim)
        dec_output, _ = self.decoder(dec_emb, hidden)  # (batch, seq_len, hidden_dim)

        output = self.fc_out(dec_output)        # (batch, seq_len, vocab_size)
        return output

rnn_model = RNNSeq2Seq().to(device)
print(f'RNN parameters: {sum(p.numel() for p in rnn_model.parameters()):,}')

In [ ]:
rnn_losses, rnn_time = train_model(rnn_model, train_loader, epochs=50, label='RNN')

In [ ]:
rnn_token_acc, rnn_seq_acc = evaluate_model(rnn_model, test_loader)
print(f'RNN — Token accuracy: {rnn_token_acc:.1%} | Sequence accuracy: {rnn_seq_acc:.1%}\n')

show_predictions(rnn_model, test_dataset)

---
## Part 4: LSTM Seq2Seq

Same encoder-decoder architecture, but we swap `nn.RNN` for `nn.LSTM`.

The LSTM adds **gates** (forget, input, output) that control information flow. Think of it as
upgrading the telephone game: instead of just whispering the whole message, each person has a
notebook (cell state) where they can write things down and selectively forget or add information.

The code change is minimal — `nn.RNN` -> `nn.LSTM` — but the LSTM's cell state gives it
better memory for longer sequences.

In [ ]:
class LSTMSeq2Seq(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden_dim=64, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Only change: nn.RNN -> nn.LSTM
        self.encoder = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)

        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src, tgt):
        src_emb = self.embedding(src)
        _, (hidden, cell) = self.encoder(src_emb)  # LSTM returns (hidden, cell) — RNN only returns hidden

        batch_size = tgt.size(0)
        sos = torch.zeros(batch_size, 1, dtype=torch.long, device=tgt.device)
        dec_input = torch.cat([sos, tgt[:, :-1]], dim=1)

        dec_emb = self.embedding(dec_input)
        dec_output, _ = self.decoder(dec_emb, (hidden, cell))  # pass both hidden AND cell state

        output = self.fc_out(dec_output)
        return output

lstm_model = LSTMSeq2Seq().to(device)
print(f'LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')
print(f'  (more params than RNN because each gate has its own weights)')

In [ ]:
lstm_losses, lstm_time = train_model(lstm_model, train_loader, epochs=50, label='LSTM')

In [ ]:
lstm_token_acc, lstm_seq_acc = evaluate_model(lstm_model, test_loader)
print(f'LSTM — Token accuracy: {lstm_token_acc:.1%} | Sequence accuracy: {lstm_seq_acc:.1%}\n')

show_predictions(lstm_model, test_dataset)

# Quick comparison so far
plt.figure(figsize=(8, 4))
plt.plot(rnn_losses, label='RNN', alpha=0.8)
plt.plot(lstm_losses, label='LSTM', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('RNN vs LSTM Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 5: Transformer (From Scratch)

Now the main event. We'll build every component from the paper:

1. **Positional Encoding** — since there's no recurrence, we inject position information with sine/cosine waves
2. **Scaled Dot-Product Attention** — the core: Q*K^T / sqrt(d_k), then softmax, then * V
3. **Multi-Head Attention** — run attention multiple times in parallel with different projections
4. **Transformer Block** — attention + add&norm + feed-forward + add&norm
5. **Full Encoder-Decoder** — stack blocks, add decoder masking

The key insight: unlike RNN/LSTM, the Transformer sees **all positions at once**.
For sequence reversal, it can directly learn "output position 0 = input position 9".

In [ ]:
# === Component 1: Positional Encoding ===
#
# From the paper (Section 3.5):
#   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
#
# Each position gets a unique pattern of sine/cosine values.
# The model learns to use these patterns to understand "where" a token is.

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)       # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)

        # The divisor term: 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dimensions

        pe = pe.unsqueeze(0)  # (1, max_len, d_model) — ready to broadcast over batches
        self.register_buffer('pe', pe)  # not a trainable parameter

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

# Visualize: each position has a unique pattern
pe_demo = PositionalEncoding(d_model=32)
pe_values = pe_demo.pe[0, :SEQ_LEN, :].numpy()

plt.figure(figsize=(10, 3))
plt.imshow(pe_values, aspect='auto', cmap='RdBu')
plt.xlabel('Embedding dimension')
plt.ylabel('Position')
plt.title('Positional Encoding — each row is a unique "position fingerprint"')
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# === Component 2: Scaled Dot-Product Attention ===
#
# From the paper (Section 3.2.1):
#   Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V
#
# Q (Query): "what am I looking for?"
# K (Key):   "what do I contain?"
# V (Value): "what information do I provide?"
#
# The dot product Q*K^T measures similarity between each query and all keys.
# Dividing by sqrt(d_k) prevents the dot products from getting too large
# (which would push softmax into regions with tiny gradients).

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, heads, seq_len, d_k)
    mask: (batch, 1, 1 or seq_len, seq_len) — positions to block
    Returns: (batch, heads, seq_len, d_k), attention_weights
    """
    d_k = Q.size(-1)

    # Step 1: Q * K^T — similarity scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (batch, heads, seq_len, seq_len)

    # Step 2: Apply mask (if provided) — set masked positions to -inf so softmax gives 0
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 3: Softmax — convert scores to probabilities
    attention_weights = torch.softmax(scores, dim=-1)  # each row sums to 1

    # Step 4: Weighted sum of values
    output = torch.matmul(attention_weights, V)  # (batch, heads, seq_len, d_k)

    return output, attention_weights

print('scaled_dot_product_attention defined')
print('  Input shapes:  Q, K, V = (batch, heads, seq_len, d_k)')
print('  Output shapes: output = (batch, heads, seq_len, d_k), weights = (batch, heads, seq_len, seq_len)')

In [ ]:
# === Component 3: Multi-Head Attention ===
#
# From the paper (Section 3.2.2):
#   Instead of one attention function with d_model dimensions,
#   we project Q, K, V into h different subspaces and run attention in parallel.
#
# Why? Each head can learn to attend to different things:
#   - Head 1 might learn "look at the mirror position" (useful for reversal)
#   - Head 2 might learn "look at adjacent tokens"
#   - Head 3 might learn "look at the first/last token"
#
# After attention, we concatenate all heads and project back.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # Linear projections for Q, K, V (all at once, then split into heads)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Final projection after concatenating heads
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Project and reshape: (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Run attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # Concatenate heads: (batch, num_heads, seq_len, d_k) -> (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # Final linear projection
        output = self.W_o(attn_output)
        return output

print('MultiHeadAttention defined')
print(f'  With d_model=32, num_heads=4: each head works with d_k={32//4}')

In [ ]:
# === Component 4: Transformer Block ===
#
# Each block in the paper follows this pattern:
#   x -> [Multi-Head Attention] -> add x (residual) -> LayerNorm
#     -> [Feed-Forward Network]  -> add (residual)   -> LayerNorm
#
# The residual connections ("add x back") are crucial — they let gradients flow
# directly through the network, making deep stacks trainable.
#
# LayerNorm stabilizes training by normalizing each sample independently.

class FeedForward(nn.Module):
    """Position-wise feed-forward network (Section 3.3)."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Self-attention + residual + norm
        attn_out = self.self_attn(x, x, x, src_mask)  # Q=K=V=x (self-attention)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward + residual + norm
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)    # masked self-attention
        self.cross_attn = MultiHeadAttention(d_model, num_heads)   # cross-attention to encoder
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        # Masked self-attention (can only look at previous positions)
        self_attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_out))

        # Cross-attention (decoder attends to encoder output)
        cross_attn_out = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_out))

        # Feed-forward
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x

print('EncoderBlock and DecoderBlock defined')
print('  Encoder: self-attention -> add&norm -> FFN -> add&norm')
print('  Decoder: masked self-attn -> add&norm -> cross-attn -> add&norm -> FFN -> add&norm')

In [ ]:
# === Component 5: Full Transformer ===

class Transformer(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=32, num_heads=4,
                 d_ff=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Embeddings + positional encoding
        self.src_embedding = nn.Embedding(vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        # Encoder stack
        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Decoder stack
        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Output projection
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def make_causal_mask(self, seq_len, device):
        """Lower-triangular mask: position i can only attend to positions <= i."""
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)
        return mask  # (1, 1, seq_len, seq_len)

    def forward(self, src, tgt):
        # Shift target right (prepend SOS, drop last token) — same as RNN/LSTM
        batch_size = tgt.size(0)
        sos = torch.zeros(batch_size, 1, dtype=torch.long, device=tgt.device)
        tgt_input = torch.cat([sos, tgt[:, :-1]], dim=1)

        # Create causal mask for decoder
        tgt_mask = self.make_causal_mask(tgt_input.size(1), tgt_input.device)

        # Encode
        src_emb = self.dropout(self.pos_encoding(
            self.src_embedding(src) * math.sqrt(self.d_model)
        ))
        enc_output = src_emb
        for layer in self.encoder_layers:
            enc_output = layer(enc_output)

        # Decode
        tgt_emb = self.dropout(self.pos_encoding(
            self.tgt_embedding(tgt_input) * math.sqrt(self.d_model)
        ))
        dec_output = tgt_emb
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, tgt_mask)

        # Project to vocabulary
        output = self.fc_out(dec_output)  # (batch, seq_len, vocab_size)
        return output

transformer_model = Transformer().to(device)
print(f'Transformer parameters: {sum(p.numel() for p in transformer_model.parameters()):,}')

In [ ]:
transformer_losses, transformer_time = train_model(
    transformer_model, train_loader, epochs=50, lr=0.001, label='Transformer'
)

In [ ]:
transformer_token_acc, transformer_seq_acc = evaluate_model(transformer_model, test_loader)
print(f'Transformer — Token accuracy: {transformer_token_acc:.1%} | Sequence accuracy: {transformer_seq_acc:.1%}\n')

show_predictions(transformer_model, test_dataset)

---
## Part 6: Comparison

Let's put all three models side by side.

In [ ]:
# --- Results Table ---
results = {
    'RNN': {
        'params': sum(p.numel() for p in rnn_model.parameters()),
        'final_loss': rnn_losses[-1],
        'token_acc': rnn_token_acc,
        'seq_acc': rnn_seq_acc,
        'time': rnn_time,
    },
    'LSTM': {
        'params': sum(p.numel() for p in lstm_model.parameters()),
        'final_loss': lstm_losses[-1],
        'token_acc': lstm_token_acc,
        'seq_acc': lstm_seq_acc,
        'time': lstm_time,
    },
    'Transformer': {
        'params': sum(p.numel() for p in transformer_model.parameters()),
        'final_loss': transformer_losses[-1],
        'token_acc': transformer_token_acc,
        'seq_acc': transformer_seq_acc,
        'time': transformer_time,
    },
}

print(f'{"Model":<14} {"Params":>8} {"Loss":>8} {"Token Acc":>10} {"Seq Acc":>10} {"Time":>8}')
print('-' * 62)
for name, r in results.items():
    print(f'{name:<14} {r["params"]:>8,} {r["final_loss"]:>8.4f} {r["token_acc"]:>9.1%} {r["seq_acc"]:>9.1%} {r["time"]:>7.1f}s')

In [ ]:
# --- Loss Curves ---
plt.figure(figsize=(10, 5))
plt.plot(rnn_losses, label='RNN', linewidth=2, alpha=0.8)
plt.plot(lstm_losses, label='LSTM', linewidth=2, alpha=0.8)
plt.plot(transformer_losses, label='Transformer', linewidth=2, alpha=0.8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.title('Training Loss: RNN vs LSTM vs Transformer', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Key Takeaways

**Why the Transformer wins at sequence reversal:**

1. **Direct access to all positions** — The Transformer's attention can directly connect output position 0 to input position 9. RNNs must propagate this information through the entire hidden state chain.

2. **Parallel computation** — The Transformer processes all positions simultaneously. RNNs must go step-by-step, which is both slower and creates a longer gradient path.

3. **Multi-head attention** — Different heads can specialize: one head learns the reversal mapping, another might track token values. This division of labor is powerful.

**Connecting to the paper ("Attention Is All You Need"):**

- The paper's Table 1 showed Transformers beating RNNs on translation. Our toy task shows the same pattern.
- The paper emphasized that self-attention has O(1) sequential operations vs O(n) for RNNs — our training time difference illustrates this.
- Positional encoding was a key innovation to give position-awareness without recurrence — and for reversal specifically, it's critical.

**What this toy task doesn't show:**
- The Transformer's advantage grows *much* larger with longer sequences and bigger datasets
- Real NLP involves much larger vocabularies (30K+ tokens) and complex grammar
- The paper's results used 6 layers and d_model=512; our tiny model is a proof of concept